# Cascade prototype
SW7 project, group S7G5, AAU Copenhagen.

Every prompt goes through three tiers, and only an *unsure* prompt moves on:

1. **Rules:** a high-specificity rule hit (no limits, ignore instructions, jailbreak names; Sections 8-9 of the report) blocks the prompt; otherwise it is unsure.
2. **Char TF-IDF + calibrated SVM:** score >= the block cutoff blocks, score <= the pass cutoff passes, anything in between is unsure. With `CHAR_TIER_ENGLISH_ONLY` it only decides plain English prompts; other languages and tampered text (invisible characters, mixed alphabets) are unsure.
3. **Prompt Guard 2 (86M):** decides every prompt that is still unsure; its answer is final.

No shared database is used. Results are saved in `results/cascade/`. Tests: `python -m unittest test_cascade_unit test_cascade_real_guard -v`.

## 1. Setup
Run this notebook from `classifier-testing`. In Colab, the first cell clones the repository and moves into `classifier-testing`.

In [ ]:
import os
from pathlib import Path

if "COLAB_RELEASE_TAG" in os.environ:
    if not Path("Software7Group5").exists():
        !git clone https://github.com/KonDiamanti/Software7Group5.git
    %cd Software7Group5/classifier-testing

%pip install -q -r requirements.txt
%pip install -q langdetect

%load_ext autoreload
%autoreload 2

In [ ]:
import sys

import pandas as pd
from huggingface_hub import get_token

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from cascade import (HAS_LANGDETECT, Cascade, cutoff_sweep, evaluate, lazy_prompt_guard,
                     load_extra_training, mistakes, summary)
from classifiers import pick_device
from prompt_data import load_test_prompts, load_training_data

DEVICE, DTYPE = pick_device()
print("Device:", DEVICE)
print("Hugging Face login:", "yes" if get_token() else "no")
print("langdetect:", "yes" if HAS_LANGDETECT else "no (every prompt counts as English)")

Prompt Guard 2 (`meta-llama/Llama-Prompt-Guard-2-86M`) and HackAPrompt (`hackaprompt/hackaprompt-dataset`) are gated: accept their terms on Hugging Face and run `login()` once. Without access, set `USE_PROMPT_GUARD = False` (unsure prompts then stay `uncertain`) and `HACKAPROMPT_SAMPLE = 0`.

**Two versions of the char tier.**
- `MORE_TRAINING_DATA = True` (default): trained on safe-guard plus half of the deepset and jackhhao *train* splits. The other half (English only) sets the two cutoffs with the 2% rule: at most 2% attacks among the passed and at most 2% harmless among the blocked prompts. The test splits are never used. Because the model has seen other prompts of deepset and jackhhao, its numbers on these two test sets are not comparable with Section 3 of the report; HackAPrompt and our own prompts stay unseen.
- `MORE_TRAINING_DATA = False`: trained on safe-guard only, as all classifiers in Section 3. Cutoffs picked from safe-guard collapse (the model is almost perfect on it, section 4 shows this), so the band is the fixed `PASS_CUTOFF` / `BLOCK_CUTOFF`.

Section 6 shows how the band trades guard calls for detection.

In [ ]:
# from huggingface_hub import login
# login()

USE_PROMPT_GUARD = True
HACKAPROMPT_SAMPLE = 500                # 0 = leave HackAPrompt out

MORE_TRAINING_DATA = True               # + deepset/jackhhao train splits, cutoffs by the 2% rule (see above)
CHAR_TIER_ENGLISH_ONLY = True           # non-English and tampered prompts skip the char tier -> Prompt Guard 2
PASS_CUTOFF, BLOCK_CUTOFF = 0.05, 0.95  # only used when MORE_TRAINING_DATA = False
GUARD_CUTOFF = 0.5                      # Prompt Guard 2 score >= this blocks
SOFT_RULES_NEED_GUARD = False           # True: a role-play / system-prompt hit is never passed by the char tier
RUN_SWEEP = True                        # section 6: scores every prompt with Prompt Guard 2 once

## 2. Data
Test prompts: deepset, jackhhao, the HackAPrompt sample and our own prompts. Training data for the char tier: safe-guard, and with `MORE_TRAINING_DATA` the deepset and jackhhao train splits, all with near copies of test prompts removed.

In [ ]:
data = load_test_prompts(HACKAPROMPT_SAMPLE)
display(data.groupby(["dataset", "label"], sort=False).size().unstack(fill_value=0)
        .rename(columns={0: "harmless", 1: "attacks"}))

safe_guard = load_training_data(data["text"])
train = safe_guard.to_pandas() if hasattr(safe_guard, "to_pandas") else safe_guard
extra = load_extra_training(data["text"]) if MORE_TRAINING_DATA else None

## 3. Prompt Guard 2
Loaded and warmed up here, so a missing licence fails now and not halfway through the run. Prompts longer than 512 tokens are checked in overlapping 512-token windows and the highest window score is used.

In [ ]:
guard = None
if USE_PROMPT_GUARD:
    if not get_token():
        raise RuntimeError("Prompt Guard 2 needs a Hugging Face login: run login() above, or set USE_PROMPT_GUARD = False.")
    guard = lazy_prompt_guard(DEVICE)
    try:
        guard("Hello, how are you?")
    except Exception as error:
        raise RuntimeError("Prompt Guard 2 could not be loaded. Accept the model terms on Hugging Face "
                           "and check the login, or set USE_PROMPT_GUARD = False.") from error
    print("Prompt Guard 2 loaded.")
else:
    print("Prompt Guard 2 is disabled; unsure prompts stay 'uncertain'.")

## 4. Char tier

In [ ]:
cascade = Cascade(guard_check=guard, pass_cutoff=PASS_CUTOFF, block_cutoff=BLOCK_CUTOFF,
                  guard_cutoff=GUARD_CUTOFF, soft_rules_need_guard=SOFT_RULES_NEED_GUARD,
                  char_tier_english_only=CHAR_TIER_ENGLISH_ONLY)
training_seconds = cascade.train(train, extra_df=extra)
print("Char TF-IDF + SVM training seconds:", round(training_seconds, 2))
print(f"Pass cutoff: {cascade.pass_cutoff:.4f}   Block cutoff: {cascade.block_cutoff:.4f}")
print("Cutoffs:", cascade.cutoff_info)

For comparison: the cutoffs that the safe-guard training data alone would pick with the same 2% rule (out-of-fold scores). The band collapses to `min_gap`, which is why that version needs a fixed band.

In [ ]:
picked = Cascade()
picked.train(train, pick=True)
print(f"Picked from safe-guard alone: pass {picked.pass_cutoff:.4f}, block {picked.block_cutoff:.4f} "
      f"(band width {picked.block_cutoff - picked.pass_cutoff:.4f})")

## 5. Run the cascade
`uncertain` counts prompts no tier decided (only without Prompt Guard 2); an uncertain attack counts as missed. `guard_calls` is how many prompts reached Prompt Guard 2.

In [ ]:
predictions = evaluate(cascade, data)
results = summary(predictions)
display(results.round(3))

### Where the prompts are decided

In [ ]:
display(pd.crosstab([predictions["dataset"], predictions["label"].map({0: "harmless", 1: "attack"})],
                    [predictions["stage"], predictions["decision"]]))
print("Prompts that skipped the char tier:", predictions["note"].replace("", "-").value_counts().to_dict())

## 6. Unsure band trade-off
Every prompt is scored once by the char tier and by Prompt Guard 2, then the cascade is replayed for several bands and for the cascade's own cutoffs. The first row is the cascade without Prompt Guard 2, the last row sends every prompt the rules do not block to Prompt Guard 2. Prompts that skip the char tier always go to Prompt Guard 2. This uses the test prompts, so use it to report the cost/benefit, not to choose the band.

In [ ]:
sweep = None
if RUN_SWEEP and guard is not None:
    sweep = cutoff_sweep(cascade, data)
    display(sweep.round(3))
else:
    print("Skipped (needs Prompt Guard 2 and RUN_SWEEP = True).")

## 7. Look at the mistakes
Change the dataset (`None` for all) and `kind` (`"missed"` or `"false_alarms"`).

In [ ]:
with pd.option_context("display.max_colwidth", 200):
    display(mistakes(predictions, "deepset", kind="missed").head(20))
    display(mistakes(predictions, None, kind="false_alarms"))

## 8. Save
Writes `results/cascade/cascade_predictions.csv` (every prompt with its decision, stage, scores and times), `cascade_results.csv` (the table in section 5) and `cascade_sweep.csv` (section 6). Every run overwrites these three files.

In [ ]:
RESULT_DIR = ROOT / "results" / "cascade"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
run_name = "cascade"

predictions.to_csv(RESULT_DIR / f"{run_name}_predictions.csv", index=False)
results.to_csv(RESULT_DIR / f"{run_name}_results.csv", index=False)
if sweep is not None:
    sweep.to_csv(RESULT_DIR / f"{run_name}_sweep.csv", index=False)
print("Saved to", RESULT_DIR / run_name, "(_predictions, _results, _sweep)")